# Surface Code Simulation — How It Works

This notebook is a guided tour of `surface_code_sim.py`.  
It does **not** run large sweeps — every cell is a small, readable demo.

**What you will learn:**
1. What a surface code is and what the simulator is measuring
2. How `ErrorModel` encodes noise
3. How Stim builds a noisy circuit from a single call
4. What *detectors* are and what the Detector Error Model (DEM) is
5. How PyMatching decodes using the DEM
6. The full `run()` pipeline, traced step by step
7. How logical error rate (and its uncertainty) are computed
8. What the sweep helpers do and when to use each one
9. How to plug in your own decoder

**Prerequisites:** basic quantum error correction concepts (stabilisers, Pauli errors, syndromes).

In [ ]:
import numpy as np
import stim
import pymatching

from surface_code_sim import (
    CodeType, ErrorModel, Decoder,
    PyMatchingDecoder, SimulationResult,
    SurfaceCodeSimulator, threshold_sweep,
)

SEED = 0

---
## 1. What is the simulator measuring?

A **logical idle experiment** works like this:

```
  prepare logical |0⟩
       │
       ▼
  ┌─────────────────┐
  │  r rounds of    │  ← physical errors happen here
  │  syndrome       │
  │  extraction     │
  └─────────────────┘
       │
       ▼
  measure logical Z
       │
       ▼
  did the decoder predict the right outcome?
```

**Logical error rate (LER)** = fraction of shots where the decoder guessed wrong.

The key numbers controlled by the simulator:

| Parameter | Meaning |
|---|---|
| `distance` d | Side length of the code lattice; code can correct up to ⌊d/2⌋ errors |
| `rounds` r | How many syndrome extraction cycles to run |
| `p_phys` | Probability of a depolarising error after every gate |
| `p_meas` | Probability of a bit-flip on every measurement |
| `p_reset` | Probability of a bit-flip after every reset (defaults to `p_meas` if not set) |
| `shots` | Number of independent Monte Carlo samples |

---
## 2. ErrorModel — encoding the noise

`ErrorModel` is just a dataclass holding two floats.  
It has three convenience constructors for common regimes.

In [ ]:
# Symmetric: gate and measurement noise equal
em_sym  = ErrorModel.symmetric(0.01)
print('Symmetric:       ', em_sym)

# Gate-dominated: measurement noise 10x smaller
em_gate = ErrorModel.gate_dominated(p_phys=0.01, meas_ratio=0.1)
print('Gate-dominated:  ', em_gate)

# Measurement-dominated: gate noise 10x smaller
em_meas = ErrorModel.meas_dominated(p_meas=0.01, gate_ratio=0.1)
print('Meas-dominated:  ', em_meas)

# You can also build one directly
em_manual = ErrorModel(p_phys=0.005, p_meas=0.003)
print('Manual:          ', em_manual)

---
## 3. CodeType — which surface code variant?

Stim has four built-in surface code recipes, selected by the string passed to `stim.Circuit.generated()`.

| `CodeType` | Stim recipe string | Notes |
|---|---|---|
| `ROTATED_Z` | `surface_code:rotated_memory_z` | Tilted lattice, logical Z, **default** |
| `ROTATED_X` | `surface_code:rotated_memory_x` | Tilted lattice, logical X |
| `UNROTATED_Z` | `surface_code:unrotated_memory_z` | Axis-aligned lattice, more qubits |
| `UNROTATED_X` | `surface_code:unrotated_memory_x` | Axis-aligned lattice, logical X |

The *rotated* code encodes the same distance with roughly **half** the physical qubits compared to the unrotated layout, which is why it is the default.

**Note on `len(circuit)`:** Stim batches all same-type gates in one layer into a single instruction line, so `CX 0 1 2 3 4 5` counts as 1 even though it applies 3 CNOT gates. The cell below counts *individual gate applications* instead, which is the true measure of work.

In [ ]:
TWO_QUBIT_GATES = {'CX', 'CZ', 'CNOT', 'CY', 'ISWAP', 'SQRT_XX', 'SQRT_YY', 'SQRT_ZZ'}

for ct in CodeType:
    sim = SurfaceCodeSimulator(distance=3, code_type=ct)
    circuit = sim.build_circuit(ErrorModel(0, 0), rounds=1)

    one_q_gates = two_q_gates = 0
    for instr in circuit:
        name = instr.name
        n_targets = len(instr.targets_copy())
        if name in TWO_QUBIT_GATES:
            two_q_gates += n_targets // 2   # each pair = 1 gate
        elif name not in ('DETECTOR', 'OBSERVABLE_INCLUDE', 'QUBIT_COORDS', 'TICK'):
            one_q_gates += n_targets        # each target = 1 gate

    print(f'{ct.name:15s}  qubits={circuit.num_qubits:3d}  '
          f'1q_gates={one_q_gates:4d}  2q_gates={two_q_gates:3d}  '
          f'instructions(lines)={len(circuit)}')

---
## 4. build_circuit() — how Stim generates the circuit

`SurfaceCodeSimulator.build_circuit()` is a thin wrapper around `stim.Circuit.generated()`.  
It passes three noise parameters:

| Stim parameter | Controlled by | Applied |
|---|---|---|
| `after_clifford_depolarization` | `p_phys` | After every 1- or 2-qubit gate |
| `before_measure_flip_probability` | `p_meas` | Before every measurement |
| `after_reset_flip_probability` | `p_reset` | After every reset (defaults to `p_meas` if not set) |

Keeping `p_reset=None` reproduces the old symmetric behaviour where measurement and reset noise are equal.

In [ ]:
# Build a tiny noiseless circuit to see the bare structure
sim3 = SurfaceCodeSimulator(distance=3)
noiseless = sim3.build_circuit(ErrorModel(0, 0), rounds=1)

print(f'Total instructions: {len(noiseless)}')
print(f'Physical qubits:    {noiseless.num_qubits}')
print()
print(noiseless)

In [ ]:
# Now add noise and see how DEPOLARIZE / X_ERROR instructions appear
noisy = sim3.build_circuit(ErrorModel.symmetric(0.01), rounds=1)

print(f'Total instructions: {len(noisy)}  (more because noise ops are inserted)')
print()
print(noisy)

### What you see in the circuit

- **`R` / `RX`** — reset ancilla qubits to |0⟩ (or |+⟩ for X-type checks)
- **`X_ERROR(p)`** — the `after_reset_flip_probability` noise
- **`H`** — Hadamard, used to enter/exit X-basis for X-type stabilisers
- **`CNOT` / `CX`** — entangle ancilla with data qubits to extract stabiliser parity
- **`DEPOLARIZE1/2(p)`** — isotropic Pauli noise after each gate
- **`M` / `MX`** — measure ancillas; result = syndrome bit
- **`DETECTOR`** — Stim meta-instruction: marks that a measurement is a *detector*
- **`OBSERVABLE_INCLUDE`** — marks measurements that contribute to the logical observable

A **detector** is a group of measurements whose XOR should be 0 in a noiseless circuit.  
When it is 1, a syndrome fires — an error was detected.

---
## 5. Detectors and the Detector Error Model (DEM)

A circuit has raw measurements.  Stim's `DETECTOR` instruction groups measurements into **detectors** — each detector XORs a set of measurements and should be 0 absent errors.

The **Detector Error Model** (DEM) is a graph where:
- each **node** is a detector
- each **edge** is a physical error mechanism with a probability and the set of detectors it flips
- edges can also be labelled with which logical observables they flip

PyMatching reads this graph directly — no hand-crafting of weights needed.

In [ ]:
dem = noisy.detector_error_model(decompose_errors=True)

print(f'Detectors:    {dem.num_detectors}')
print(f'Observables:  {dem.num_observables}')
print()
# Print first 30 lines of the DEM to understand its structure
dem_str = str(dem).split('\n')
for line in dem_str[:30]:
    print(line)
if len(dem_str) > 30:
    print(f'... ({len(dem_str) - 30} more lines)')

### Reading the DEM

Each line has the form:
```
error(p)  D<i>  D<j>  L<k>
```
- `p` — probability this error mechanism fires
- `D<i>`, `D<j>` — the detectors this error flips (the syndrome it causes)
- `L<k>` — if present, this error also flips logical observable k

A single-qubit Pauli on a data qubit typically flips 1–2 neighbouring detectors.  
A measurement error flips the detector for that round and the next (since measurements appear in two consecutive detectors).

---
## 6. PyMatchingDecoder — how MWPM decoding works

**Minimum-Weight Perfect Matching (MWPM)** solves:
> Given a set of fired detectors, find the lowest-probability set of error mechanisms that explains them.

Steps:
1. Build a graph from the DEM (nodes = detectors, edge weight = −log p)
2. From the sample, collect all detectors that fired (value = 1)
3. Find a perfect matching on those nodes — pairs them up with minimum total weight
4. XOR the logical labels of matched edges to get the predicted logical correction

`PyMatchingDecoder.setup()` does step 1 once; `decode_batch()` does steps 2–4 for every shot.

In [ ]:
decoder = PyMatchingDecoder()
decoder.setup(noisy)

# Peek at the matching object
m = decoder._matching
print(f'Matching graph: {m.num_nodes} nodes, {m.num_edges} edges')

# Sample a few detection events manually to see what the decoder receives
sampler = noisy.compile_detector_sampler(seed=SEED)
detection_events, observable_flips = sampler.sample(5, separate_observables=True)

print(f'\ndetection_events shape: {detection_events.shape}  (shots × detectors)')
print(f'observable_flips shape: {observable_flips.shape}  (shots × observables)')
print()
print('detection_events (first 5 shots):')
print(detection_events.astype(int))
print()
print('observable_flips (ground truth):')
print(observable_flips.astype(int))

In [ ]:
# Decode those 5 shots
predictions = decoder.decode_batch(detection_events)
print('predictions (decoder output):')
print(predictions.astype(int))

print()
# A shot is a logical error when any observable is mispredicted
logical_errors = np.any(predictions != observable_flips, axis=1)
print('logical error per shot:', logical_errors.astype(int))
print('(1 = decoder was wrong, 0 = correct)')

---
## 7. The full run() pipeline — step by step

Here is the exact sequence inside `SurfaceCodeSimulator.run()`, executed manually so you can inspect each step.

In [ ]:
# ── Step 1: build the noisy circuit ──────────────────────────────────────────
sim = SurfaceCodeSimulator(distance=3)
em  = ErrorModel.symmetric(0.01)
rounds = 3

circuit = sim.build_circuit(em, rounds)
print(f'Step 1 — circuit has {circuit.num_qubits} qubits, '
      f'{circuit.num_detectors} detectors, '
      f'{circuit.num_observables} observable(s)')

In [ ]:
# ── Step 2: set up the decoder (builds matching graph from DEM) ───────────────
decoder = PyMatchingDecoder()
decoder.setup(circuit)
print('Step 2 — decoder ready')

In [ ]:
# ── Step 3: sample shots ──────────────────────────────────────────────────────
shots = 2_000
sampler = circuit.compile_detector_sampler(seed=SEED)
detection_events, observable_flips = sampler.sample(shots, separate_observables=True)

print(f'Step 3 — sampled {shots} shots')
print(f'  detection_events: {detection_events.shape}  (bool array)')
print(f'  observable_flips: {observable_flips.shape}')
print(f'  syndrome sparsity: {detection_events.mean():.3f} '
      f'(fraction of detectors that fire per shot)')

In [ ]:
# ── Step 4: decode ────────────────────────────────────────────────────────────
predictions = decoder.decode_batch(detection_events)
print(f'Step 4 — decoded, predictions shape: {predictions.shape}')

In [ ]:
# ── Step 5: compute logical error rate ───────────────────────────────────────
#   A shot is a logical error if ANY observable is mispredicted.
logical_errors_per_shot = np.any(predictions != observable_flips, axis=1)
n_err = int(np.sum(logical_errors_per_shot))
ler   = n_err / shots
ler_se = float(np.sqrt(ler * (1.0 - ler) / shots))  # binomial SE

print(f'Step 5 — results')
print(f'  Logical errors: {n_err} / {shots}')
print(f'  LER = {ler:.4f} ± {ler_se:.4f}')
print()
# This matches what run() returns:
result = sim.run(em, rounds=rounds, shots=shots, seed=SEED)
print('run() returns:', result)

---
## 8. Standard error and shot count

The logical error rate is estimated from a binomial proportion, so its standard error is:

$$\text{SE} = \sqrt{\frac{\hat{p}(1-\hat{p})}{N}}$$

**Rule of thumb:** to halve the uncertainty, quadruple the shots.  
Near the threshold (~1% error rate), 10 000 shots gives SE ≈ 0.001.  
For a 5-sigma claim you want the signal ≫ SE.

In [ ]:
# Show how SE shrinks with more shots for a fixed LER
true_ler = 0.01
for n in [100, 1_000, 10_000, 100_000]:
    se = np.sqrt(true_ler * (1 - true_ler) / n)
    rel = se / true_ler * 100
    print(f'shots={n:7d}  SE={se:.5f}  relative={rel:.1f}%')

---
## 9. The sweep helpers

### 9a. sweep_p — vary physical error rate at fixed distance & rounds

In [ ]:
# sweep_p runs sim.run() for each p value and collects results
sim5 = SurfaceCodeSimulator(distance=5)
p_values = [0.001, 0.003, 0.005, 0.010]

results = sim5.sweep_p(p_values, rounds=5, shots=2_000, seed=SEED)
for r in results:
    print(r)

### 9b. sweep_rounds — vary rounds at fixed error rate

In [ ]:
em_fixed = ErrorModel.symmetric(0.005)
round_values = [1, 3, 5, 7, 10]

results_rounds = sim5.sweep_rounds(round_values, em_fixed, shots=2_000, seed=SEED)
for r in results_rounds:
    print(f'rounds={r.rounds:3d}  LER={r.logical_error_rate:.4f} ± {r.logical_error_rate_se:.4f}')

### 9c. threshold_sweep — vary both distance and error rate

This produces the crossing-curve plot used to locate the error threshold.  
Internally it creates a `SurfaceCodeSimulator` per distance and calls `sweep_p` on each.

In [ ]:
sweep = threshold_sweep(
    distances=[3, 5],
    p_values=[0.003, 0.007, 0.012, 0.020],
    shots=1_000,
    seed=SEED,
)

for d, results in sweep.items():
    print(f'\nd={d}')
    for r in results:
        print(f'  p={r.error_model.p_phys:.3f}  LER={r.logical_error_rate:.4f}')

---
## 10. Writing your own decoder

Any class that subclasses `Decoder` and implements `setup()` and `decode_batch()` can be passed to `run()` or `sweep_p()`.

### Interface contract

```python
class Decoder:
    def setup(self, circuit: stim.Circuit) -> None:
        # Called once. Use the circuit's DEM to pre-compute
        # whatever graph/data structure your decoder needs.
        ...

    def decode_batch(self, detection_events: np.ndarray) -> np.ndarray:
        # detection_events: bool array of shape (shots, num_detectors)
        # Return:           bool array of shape (shots, num_observables)
        #   predictions[i, k] == True means "logical observable k was flipped"
        ...
```

Below is a **lookup table decoder** that caches the MWPM answer for each unique syndrome.  
It trades memory for speed when the same syndromes repeat often (small code, many shots).

In [ ]:
class CachedMatchingDecoder(Decoder):
    """
    MWPM decoder with a syndrome → correction cache.
    Speeds up repeated syndromes at the cost of memory.
    """

    def setup(self, circuit: stim.Circuit) -> None:
        dem = circuit.detector_error_model(decompose_errors=True)
        self._matching = pymatching.Matching.from_detector_error_model(dem)
        self._cache: dict = {}

    def decode_batch(self, detection_events: np.ndarray) -> np.ndarray:
        shots, n_det = detection_events.shape
        n_obs = self._matching.num_fault_ids
        predictions = np.zeros((shots, n_obs), dtype=bool)

        for i, syndrome in enumerate(detection_events):
            key = syndrome.tobytes()
            if key not in self._cache:
                single = syndrome.reshape(1, -1)
                self._cache[key] = self._matching.decode_batch(single)[0]
            predictions[i] = self._cache[key]

        return predictions


# Compare standard PyMatching vs cached version
em_test = ErrorModel.symmetric(0.01)
sim_test = SurfaceCodeSimulator(distance=3)

r_std    = sim_test.run(em_test, rounds=3, shots=3_000, decoder=PyMatchingDecoder(), seed=SEED)
r_cached = sim_test.run(em_test, rounds=3, shots=3_000, decoder=CachedMatchingDecoder(), seed=SEED)

print('Standard PyMatching: ', r_std)
print('Cached PyMatching:   ', r_cached)
print('(Results differ only by sampling noise, not decoding quality)')

---
## Summary

| Component | File location | What it does |
|---|---|---|
| `CodeType` | `surface_code_sim.py:25` | Enum of Stim recipe strings |
| `ErrorModel` | `surface_code_sim.py:37` | Holds `p_phys`, `p_meas`; 3 constructors |
| `Decoder` | `surface_code_sim.py:83` | Abstract interface: `setup()` + `decode_batch()` |
| `PyMatchingDecoder` | `surface_code_sim.py:105` | Wraps PyMatching MWPM |
| `SimulationResult` | `surface_code_sim.py:131` | Dataclass: LER ± SE + metadata |
| `SurfaceCodeSimulator` | `surface_code_sim.py:155` | Builds circuits, runs experiments, sweeps |
| `threshold_sweep` | `surface_code_sim.py:317` | Multi-distance p-sweep for threshold plots |

**Data flow:**
```
ErrorModel + CodeType
       │
       ▼
 build_circuit()  →  stim.Circuit
                          │
            ┌─────────────┴──────────────┐
            ▼                            ▼
     decoder.setup()          compile_detector_sampler()
     (DEM → graph)                       │
            │                    sample(shots)
            │                            │
            └──── decode_batch(detection_events) ────▶ predictions
                                                           │
                                              compare to observable_flips
                                                           │
                                                  SimulationResult
```